<a href="https://colab.research.google.com/github/Kayla-afk/Tugas-Kuliah-D4-Sains-Data-Terapan/blob/main/DM_M6_3324600023_Kayla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup dan Load Dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Analisa
Pada tahap ini dilakukan:
* Import library utama (pandas, numpy) untuk manipulasi data
* Upload dataset Titanic ke environment Colab
* Membaca file CSV menjadi DataFrame
Tahap ini penting dilakukan karena seluruh proses nantinya akan bergantung pada struktur data yang benar.

### Seleksi Fitur dan Label

In [3]:
#Pilih Fitur
train_data = df[['Sex', 'Age', 'Pclass', 'Fare']]

#Label
label = df['Survived']

## Analisa
Dilakukan pemisahan:
* Fitur (X) → variabel independen: Sex, Age, Pclass, Fare
* Label (y) → variabel target: Survived

Pemilihan fitur ini sesuai instruksi soal dan merepresentasikan:
* Demografi (Sex, Age)
* Kelas sosial (Pclass)
* Ekonomi (Fare)

### Preprocessing

In [4]:
#1. Encoding fitur kategorikal (Sex -> numerik)
train_data['Sex'] = train_data['Sex'].map({'male':0, 'female':1})

/tmp/ipykernel_30426/1602515094.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['Sex'] = train_data['Sex'].map({'male':0, 'female':1})


## Analisa
Algoritma KNN hanya menerima data numerik, sehingga:

* Sex dikonversi:
male → 0
female → 1

Transformasi ini bertujuan menjaga kategori tanpa menambah kompleksitas dimensi.

In [5]:
#2. Handling Missing Value (Age -> mean per class Pclass)
#Isi missing value Age berdasarkan mean tiap Pclass
train_data['Age'] = train_data.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.mean()))

/tmp/ipykernel_30426/2967133386.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['Age'] = train_data.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.mean()))


## Analisa
Fitur Age memiliki missing value. Penanganan dilakukan dengan:

* Menggunakan mean per Pclass
* Bukan mean global

Alasan:

* Usia penumpang cenderung berkorelasi dengan kelas sosial
* Mengurangi bias imputasi

### Normalisasi (Min-Max dari Train Data)

In [6]:
#Simpan min dan max dari train_data
min_vals = train_data.min()
max_vals = train_data.max()

#Normalisasi train_data
train_data_norm = (train_data - min_vals) / (max_vals - min_vals)

## Analisa
Normalisasi diperlukan karena:

* KNN berbasis jarak (distance-based)
* Skala fitur berbeda (Fare bisa sangat besar dibanding Age)

Keterangan:
* Min & Max diambil dari train_data
* Digunakan konsisten untuk semua proses validasi

### Model KNN

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## Analisa
* KNeighborsClassifier → algoritma klasifikasi berbasis jarak
* accuracy_score tidak dipakai karena soal meminta error ratio


### Validation Methods

In [8]:
#A. Hold-Out (70:30)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    train_data_norm, label, test_size=0.3, random_state=42
)

#Model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

#Prediksi
y_pred = knn.predict(X_test)

#Error Ratio
error_ratio = np.mean(y_pred !=y_test)

print("Hold-Out Error Ratio:", error_ratio)

Hold-Out Error Ratio: 0.1865671641791045


## Analisa
Metode ini betujuan untuk membagi data menjadi train (70%) dan test (30%) dengan komputasi cepat namun sensitif terhadap pembagian data.

Langkah:
* Split data
* Train model
* Prediksi
* Hitung error ratio

In [10]:
#B. K-Fold Cross Validation (k=10)
from sklearn.model_selection import KFold

kf = KFold(n_splits=10, shuffle=True, random_state=42)

errors = []
for train_index, test_index in kf.split(train_data_norm):
  X_train, X_test = train_data_norm.iloc[train_index], train_data_norm.iloc[test_index]
  y_train, y_test = label.iloc[train_index], label.iloc[test_index]

  model = KNeighborsClassifier(n_neighbors=3)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

  error = np.mean(y_pred !=y_test)
  errors.append(error)

print("K-Fold Error Ratio:", np.mean(errors))

K-Fold Error Ratio: 0.17508114856429463


## Analisa
Metode ini:
* Membagi data menjadi 10 fold
* Setiap fold menjadi test secara bergantian
* Lebih stabil dibanding hold-out

Proses:
* Loop seluruh fold
* Hitung error tiap fold
* Ambil rata-rata error

In [11]:
#C. Leave-One-Out (LOO)
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
errors = []

for train_index, test_index in loo.split(train_data_norm):
    X_train, X_test = train_data_norm.iloc[train_index], train_data_norm.iloc[test_index]
    y_train, y_test = label.iloc[train_index], label.iloc[test_index]

    model = KNeighborsClassifier(n_neighbors=3)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    error = np.mean(y_pred != y_test)
    errors.append(error)

print("LOO Error Ratio:", np.mean(errors))

LOO Error Ratio: 0.1728395061728395


## Analisa
LOO adalah kasus yang lebih kompleks dari K-Fold:
* 1 data sebagai test
* sisanya sebagai train
* diulang sebanyak jumlah data

Kelebihan:
* Hampir tidak bias

Kekurangan:
* Sangat mahal secara komputasi

* Analisa Hasil
* Perbandingan metode:
  * Hold-Out → cepat, kurang stabil
  * K-Fold → seimbang (recommended)
  * LOO → paling akurat secara teori, tapi mahal